In [2]:
print('Movie Recommender')

Movie Recommender


In [3]:
import pandas as pd
from pathlib import Path

# Locate the dataset relative to the project folder
data_path = Path("../data/raw/movielens_100k.csv")

movies_df = pd.read_csv(data_path)

print("Dataset shape:", movies_df.shape)
movies_df.head()

Dataset shape: (1681, 6)


,movie_id,title,year,directors,actors,genres
0,1,toy story,1995,John Lasseter,Tom Hanks Tim Allen Don Rickles Jim Varney Wal...,Animation Adventure Comedy Family Fantasy
1,2,goldeneye,1995,Martin Campbell,Pierce Brosnan Sean Bean Izabella Scorupco Fam...,Action Adventure Thriller
2,3,four rooms,1995,Allison Anders Alexandre Rockwell Robert Rodri...,Sammi Davis Amanda De Cadenet Valeria Golino M...,Comedy
3,4,get shorty,1995,Barry Sonnenfeld,John Travolta Gene Hackman Rene Russo Danny De...,Comedy Crime Thriller
4,5,copycat,1995,Jon Amiel,Sigourney Weaver Holly Hunter Dermot Mulroney ...,Drama Mystery Thriller


In [4]:
print("Column names:")
print(movies_df.columns.tolist())

print("\nData types and non-null counts:")
movies_df.info()

print("\nMissing values per column:")
print(movies_df.isnull().sum())

Column names:
['movie_id', 'title', 'year', 'directors', 'actors', 'genres']

Data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 1681 entries, 0 to 1680
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   movie_id   1681 non-null   int64
 1   title      1681 non-null   str  
 2   year       1681 non-null   int64
 3   directors  1555 non-null   str  
 4   actors     1555 non-null   str  
 5   genres     1561 non-null   str  
dtypes: int64(2), str(4)
memory usage: 78.9 KB

Missing values per column:
movie_id       0
title          0
year           0
directors    126
actors       126
genres       120
dtype: int64


In [5]:
# Count missing values in each column
movies_df.isnull().sum()

movie_id       0
title          0
year           0
directors    126
actors       126
genres       120
dtype: int64

In [6]:
# Check for duplicate movie IDs and titles
print("Duplicate movie IDs:", movies_df["movie_id"].duplicated().sum())
print("Duplicate titles:", movies_df["title"].duplicated().sum())

# Inspect how many unique values each column contains
print("\nUnique values:")
print(movies_df.nunique())

Duplicate movie IDs: 0
Duplicate titles: 23

Unique values:
movie_id     1681
title        1658
year           71
directors     999
actors       1531
genres        459
dtype: int64


In [7]:
# Make a working copy so we preserve the original data
movies = movies_df.copy()

# Replace missing text values with empty strings
text_columns = ["directors", "actors", "genres"]

for column in text_columns:
    movies[column] = movies[column].fillna("")

# Build a combined profile for every movie
movies["combined_features"] = (
    "genres " + movies["genres"] + " " +
    "director " + movies["directors"] + " " +
    "actors " + movies["actors"]
).str.lower()

# Inspect the result
movies[["title", "combined_features"]].head()

,title,combined_features
0,toy story,genres animation adventure comedy family fanta...
1,goldeneye,genres action adventure thriller director mart...
2,four rooms,genres comedy director allison anders alexandr...
3,get shorty,genres comedy crime thriller director barry so...
4,copycat,genres drama mystery thriller director jon ami...


In [8]:
movies.combined_features

0       genres animation adventure comedy family fanta...
1       genres action adventure thriller director mart...
2       genres comedy director allison anders alexandr...
3       genres comedy crime thriller director barry so...
4       genres drama mystery thriller director jon ami...
                              ...                        
1676    genres drama director aleksandr sokurov actors...
1677    genres crime drama romance thriller director m...
1678    genres comedy drama fantasy romance director p...
1679    genres documentary comedy director thomas schl...
1680    genres drama director werner herzog actors vit...
Name: combined_features, Length: 1681, dtype: str

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
tfidf = TfidfVectorizer(
    stop_words="english"
)

# Learn the vocabulary and transform movie profiles into vectors
tfidf_matrix = tfidf.fit_transform(movies["combined_features"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of unique terms:", len(tfidf.get_feature_names_out()))

TF-IDF matrix shape: (1681, 32176)
Number of unique terms: 32176


In [10]:
# Show the vocabulary's first 20 terms
print(tfidf.get_feature_names_out()[:20])

# Convert the first movie's vector into a readable table
import pandas as pd

first_movie_vector = tfidf_matrix[0].toarray().flatten()

movie_features = pd.DataFrame({
    "term": tfidf.get_feature_names_out(),
    "tfidf_score": first_movie_vector
})

movie_features = movie_features[
    movie_features["tfidf_score"] > 0
].sort_values("tfidf_score", ascending=False)

movie_features.head(15)

['10' '2meka' '40' '4th' 'aadil' 'aaker' 'aalam' 'aalda' 'aale'
 'aalmuhammed' 'aaltonen' 'aar' 'aaran' 'aard' 'aarie' 'aarniokoski'
 'aaron' 'aarons' 'aarón' 'aasif']


,term,tfidf_score
16538,lasseter,0.448967
11582,grudt,0.149656
1413,aste,0.149656
17003,levenbrown,0.149656
14139,jillette,0.141810
23993,ranft,0.141810
14392,judovits,0.141810
24498,rickles,0.141810
7989,dorkin,0.136244
4152,burch,0.136244


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate pairwise similarity between all movies
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)

# Show similarity scores for the first movie against the first 10 movies
print(similarity_matrix[0, :10].round(3))

Similarity matrix shape: (1681, 1681)
[1.    0.025 0.009 0.024 0.043 0.034 0.041 0.063 0.045 0.037]


In [12]:
def recommend_movies(title, top_n=10):
    # Find the selected movie
    matches = movies[movies["title"].str.casefold() == title.casefold()]

    if matches.empty:
        print(f"Movie not found: {title}")
        return None

    # Use the first matching movie if titles are duplicated
    movie_index = matches.index[0]

    # Pair each movie index with its similarity score
    scores = list(enumerate(similarity_matrix[movie_index]))

    # Sort from most similar to least similar
    scores = sorted(scores, key=lambda item: item[1], reverse=True)

    # Exclude the selected movie itself
    scores = [
        (idx, score)
        for idx, score in scores
        if idx != movie_index
    ]

    # Select the top N movies
    top_movies = scores[:top_n]

    # Return useful movie information and similarity scores
    recommendations = movies.loc[
        [idx for idx, score in top_movies],
        ["movie_id", "title", "year", "genres"]
    ].copy()

    recommendations["similarity"] = [
        score for idx, score in top_movies
    ]

    return recommendations.reset_index(drop=True)

In [13]:
recommend_movies("toy story", top_n=10)

,movie_id,title,year,genres,similarity
0,993,hercules,1997,Animation Adventure Comedy Family Fantasy Musi...,0.191165
1,542,pocahontas,1995,Animation Adventure Drama Family Musical Romance,0.184761
2,95,aladdin,1992,Animation Adventure Comedy Family Fantasy Musi...,0.168390
3,240,beavis and butthead do america,1996,Animation Adventure Comedy Crime Music,0.166090
4,1219,goofy movie a,1995,Animation Adventure Comedy Family Musical Romance,0.156056
5,596,hunchback of notre dame the,1996,Animation Drama Family Musical Romance,0.152521
6,588,beauty and the beast,1991,Animation Family Fantasy Musical Romance,0.117743
7,104,theodore rex,1995,Action Comedy Family Sci-Fi,0.103553
8,1412,land before time iii the time of the great giv...,1995,Animation Adventure Family Musical,0.091527
9,422,aladdin and the king of thieves,1996,Animation Adventure Comedy Family Fantasy Musi...,0.077942


In [14]:
movies["weighted_features"] = (
    "genres " + movies["genres"] + " " +
    "genres " + movies["genres"] + " " +
    "director " + movies["directors"] + " " +
    "actors " + movies["actors"]
).str.lower()

movies["weighted_features"]

0       genres animation adventure comedy family fanta...
1       genres action adventure thriller genres action...
2       genres comedy genres comedy director allison a...
3       genres comedy crime thriller genres comedy cri...
4       genres drama mystery thriller genres drama mys...
                              ...                        
1676    genres drama genres drama director aleksandr s...
1677    genres crime drama romance thriller genres cri...
1678    genres comedy drama fantasy romance genres com...
1679    genres documentary comedy genres documentary c...
1680    genres drama genres drama director werner herz...
Name: weighted_features, Length: 1681, dtype: str

In [15]:
tfidf_weighted = TfidfVectorizer(stop_words="english")

tfidf_matrix_weighted = tfidf_weighted.fit_transform(
    movies["weighted_features"]
)

similarity_matrix_weighted = cosine_similarity(
    tfidf_matrix_weighted
)

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_weighted = TfidfVectorizer(stop_words="english")

tfidf_matrix_weighted = tfidf_weighted.fit_transform(
    movies["weighted_features"]
)

similarity_matrix_weighted = cosine_similarity(
    tfidf_matrix_weighted
)

print("Weighted TF-IDF matrix:", tfidf_matrix_weighted.shape)
print("Weighted similarity matrix:", similarity_matrix_weighted.shape)

Weighted TF-IDF matrix: (1681, 32176)
Weighted similarity matrix: (1681, 1681)


In [17]:
def recommend_movies_weighted(title, top_n=10):
    matches = movies[
        movies["title"].str.casefold() == title.casefold()
    ]

    if matches.empty:
        print(f"Movie not found: {title}")
        return None

    movie_index = matches.index[0]

    scores = list(
        enumerate(similarity_matrix_weighted[movie_index])
    )

    scores.sort(key=lambda item: item[1], reverse=True)

    scores = [
        (idx, score)
        for idx, score in scores
        if idx != movie_index
    ]

    top_movies = scores[:top_n]

    recommendations = movies.loc[
        [idx for idx, score in top_movies],
        ["movie_id", "title", "year", "genres"]
    ].copy()

    recommendations["similarity"] = [
        score for idx, score in top_movies
    ]

    return recommendations.reset_index(drop=True)


recommend_movies_weighted("toy story", top_n=10)

,movie_id,title,year,genres,similarity
0,993,hercules,1997,Animation Adventure Comedy Family Fantasy Musi...,0.224991
1,542,pocahontas,1995,Animation Adventure Drama Family Musical Romance,0.221842
2,95,aladdin,1992,Animation Adventure Comedy Family Fantasy Musi...,0.207467
3,240,beavis and butthead do america,1996,Animation Adventure Comedy Crime Music,0.190174
4,1219,goofy movie a,1995,Animation Adventure Comedy Family Musical Romance,0.189925
5,596,hunchback of notre dame the,1996,Animation Drama Family Musical Romance,0.177051
6,1412,land before time iii the time of the great giv...,1995,Animation Adventure Family Musical,0.156256
7,588,beauty and the beast,1991,Animation Family Fantasy Musical Romance,0.147031
8,422,aladdin and the king of thieves,1996,Animation Adventure Comedy Family Fantasy Musi...,0.145112
9,1462,thieves voleurs les,1996,Animation Adventure Comedy Family Fantasy Musi...,0.145112


In [18]:
recommend_movies_weighted("toy story", top_n=10)

,movie_id,title,year,genres,similarity
0,993,hercules,1997,Animation Adventure Comedy Family Fantasy Musi...,0.224991
1,542,pocahontas,1995,Animation Adventure Drama Family Musical Romance,0.221842
2,95,aladdin,1992,Animation Adventure Comedy Family Fantasy Musi...,0.207467
3,240,beavis and butthead do america,1996,Animation Adventure Comedy Crime Music,0.190174
4,1219,goofy movie a,1995,Animation Adventure Comedy Family Musical Romance,0.189925
5,596,hunchback of notre dame the,1996,Animation Drama Family Musical Romance,0.177051
6,1412,land before time iii the time of the great giv...,1995,Animation Adventure Family Musical,0.156256
7,588,beauty and the beast,1991,Animation Family Fantasy Musical Romance,0.147031
8,422,aladdin and the king of thieves,1996,Animation Adventure Comedy Family Fantasy Musi...,0.145112
9,1462,thieves voleurs les,1996,Animation Adventure Comedy Family Fantasy Musi...,0.145112


In [19]:
tfidf_genres = TfidfVectorizer(stop_words="english")

genre_matrix = tfidf_genres.fit_transform(
    movies["genres"].fillna("").str.lower()
)

genre_similarity = cosine_similarity(genre_matrix)

print("Genre-only matrix:", genre_matrix.shape)

Genre-only matrix: (1681, 27)


In [21]:
def recommend_by_genre(title, top_n=10):
    matches = movies[
        movies["title"].str.casefold() == title.casefold()
    ]

    if matches.empty:
        print(f"Movie not found: {title}")
        return None

    movie_index = matches.index[0]

    scores = list(enumerate(genre_similarity[movie_index]))
    scores.sort(key=lambda item: item[1], reverse=True)

    scores = [
        (idx, score)
        for idx, score in scores
        if idx != movie_index
    ]

    top_movies = scores[:top_n]

    recommendations = movies.loc[
        [idx for idx, score in top_movies],
        ["movie_id", "title", "year", "genres"]
    ].copy()

    recommendations["similarity"] = [
        score for idx, score in top_movies
    ]

    return recommendations.reset_index(drop=True)


recommend_by_genre("toy story", top_n=10)

,movie_id,title,year,genres,similarity
0,1076,pagemaster the,1994,Animation Adventure Comedy Family Fantasy,1.000000
1,103,all dogs go to heaven 2,1996,Animation Adventure Family Fantasy Romance,0.918552
2,142,bedknobs and broomsticks,1971,Animation Adventure Comedy Family Fantasy Musical,0.869145
3,625,sword in the stone the,1963,Animation Adventure Comedy Family Fantasy Musical,0.869145
4,1091,petes dragon,1977,Animation Adventure Comedy Family Fantasy Musical,0.869145
5,95,aladdin,1992,Animation Adventure Comedy Family Fantasy Musi...,0.839473
6,422,aladdin and the king of thieves,1996,Animation Adventure Comedy Family Fantasy Musi...,0.839473
7,993,hercules,1997,Animation Adventure Comedy Family Fantasy Musi...,0.839473
8,1462,thieves voleurs les,1996,Animation Adventure Comedy Family Fantasy Musi...,0.839473
9,420,alice in wonderland,1951,Animation Adventure Family Fantasy Musical,0.830635


In [22]:
toy_story_index = movies[
    movies["title"].str.casefold() == "toy story"
].index[0]

models_to_compare = {
    "Original": similarity_matrix,
    "Weighted": similarity_matrix_weighted,
    "Genre-only": genre_similarity,
}

for model_name, matrix in models_to_compare.items():
    scores = list(enumerate(matrix[toy_story_index]))
    scores.sort(key=lambda item: item[1], reverse=True)

    top_movies = [
        (idx, score)
        for idx, score in scores
        if idx != toy_story_index
    ][:5]

    print(f"\n{model_name} model")
    for idx, score in top_movies:
        print(
            f"{movies.loc[idx, 'title']}: "
            f"{score:.3f}"
        )


Original model
hercules: 0.191
pocahontas: 0.185
aladdin: 0.168
beavis and butthead do america: 0.166
goofy movie a: 0.156

Weighted model
hercules: 0.225
pocahontas: 0.222
aladdin: 0.207
beavis and butthead do america: 0.190
goofy movie a: 0.190

Genre-only model
pagemaster the: 1.000
all dogs go to heaven 2: 0.919
bedknobs and broomsticks: 0.869
sword in the stone the: 0.869
petes dragon: 0.869


In [24]:
from pathlib import Path

print("Current directory:", Path.cwd())

Current directory: c:\Users\HS\OneDrive\Documents\VS Code\movie-recommender-system\notebooks


In [25]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.recommender import recommend_movies

print("Import successful!")

Import successful!


In [26]:
recommendations = recommend_movies(
    title="toy story",
    movies=movies,
    similarity_matrix=similarity_matrix,
    top_n=10
)

recommendations

,movie_id,title,year,genres,similarity
0,993,hercules,1997,Animation Adventure Comedy Family Fantasy Musi...,0.191165
1,542,pocahontas,1995,Animation Adventure Drama Family Musical Romance,0.184761
2,95,aladdin,1992,Animation Adventure Comedy Family Fantasy Musi...,0.168390
3,240,beavis and butthead do america,1996,Animation Adventure Comedy Crime Music,0.166090
4,1219,goofy movie a,1995,Animation Adventure Comedy Family Musical Romance,0.156056
5,596,hunchback of notre dame the,1996,Animation Drama Family Musical Romance,0.152521
6,588,beauty and the beast,1991,Animation Family Fantasy Musical Romance,0.117743
7,104,theodore rex,1995,Action Comedy Family Sci-Fi,0.103553
8,1412,land before time iii the time of the great giv...,1995,Animation Adventure Family Musical,0.091527
9,422,aladdin and the king of thieves,1996,Animation Adventure Comedy Family Fantasy Musi...,0.077942


In [27]:
recommend_movies(
    title="a movie that does not exist",
    movies=movies,
    similarity_matrix=similarity_matrix,
    top_n=10
)

ValueError: Movie not found: a movie that does not exist

In [28]:
try:
    recommendations = recommend_movies(
        title="a movie that does not exist",
        movies=movies,
        similarity_matrix=similarity_matrix,
        top_n=10
    )
    display(recommendations)

except ValueError as error:
    print(error)

Movie not found: a movie that does not exist


In [29]:
from src.data_loader import load_movies

movies_loaded = load_movies()

print("Dataset shape:", movies_loaded.shape)
print("Columns:", movies_loaded.columns.tolist())
print(movies_loaded.head())

Dataset shape: (1681, 6)
Columns: ['movie_id', 'title', 'year', 'directors', 'actors', 'genres']
   movie_id       title  year  \
0         1   toy story  1995   
1         2   goldeneye  1995   
2         3  four rooms  1995   
3         4  get shorty  1995   
4         5     copycat  1995   

                                           directors  \
0                                      John Lasseter   
1                                    Martin Campbell   
2  Allison Anders Alexandre Rockwell Robert Rodri...   
3                                   Barry Sonnenfeld   
4                                          Jon Amiel   

                                              actors  \
0  Tom Hanks Tim Allen Don Rickles Jim Varney Wal...   
1  Pierce Brosnan Sean Bean Izabella Scorupco Fam...   
2  Sammi Davis Amanda De Cadenet Valeria Golino M...   
3  John Travolta Gene Hackman Rene Russo Danny De...   
4  Sigourney Weaver Holly Hunter Dermot Mulroney ...   

                              

In [31]:
from src.model import build_similarity_model

tfidf_matrix_new, similarity_matrix_new = build_similarity_model(
    movies_loaded
)

print("TF-IDF matrix:", tfidf_matrix_new.shape)
print("Similarity matrix:", similarity_matrix_new.shape)

TF-IDF matrix: (1681, 32176)
Similarity matrix: (1681, 1681)
